# 06 — Script Editor

Improve a generated script for retention, clarity, rhythm, and spoken delivery.

This notebook:

1. Loads the newest script from `data/scripts/`.
2. Removes obsolete visual-direction fields through the simplified schema.
3. Sends the narration to the local LLM for editing.
4. Recalculates timing from the edited word count.
5. Saves the result to `data/edited_scripts/`.
6. Shows a before-and-after comparison.


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Find the nearest parent containing the educational_shorts package."""
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.editor import (
    build_edited_script_filename,
    edit_script,
    find_script_file,
    load_script,
    save_script,
    summarize_changes,
)
from educational_shorts.prompts import load_prompt

print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
SCRIPTS_DIRECTORY = PROJECT_ROOT / "data" / "scripts"
EDITED_SCRIPTS_DIRECTORY = PROJECT_ROOT / "data" / "edited_scripts"

# Set this to a specific JSON filename, or leave it as None to use the newest
# script file.
SCRIPT_FILENAME = None

TARGET_WORDS_PER_MINUTE = 145
MINIMUM_SECONDS = 40
MAXIMUM_SECONDS = 60
TEMPERATURE = 0.4
GENERATION_SEED = 42

print(f"Input directory:  {SCRIPTS_DIRECTORY}")
print(f"Output directory: {EDITED_SCRIPTS_DIRECTORY}")

## Load the generated script

In [ ]:
script_path = find_script_file(
    scripts_directory=SCRIPTS_DIRECTORY,
    filename=SCRIPT_FILENAME,
)

original_script = load_script(script_path)

print(f"Loaded: {script_path}")
print(f"Title: {original_script.topic.title}")
print(f"Words: {original_script.word_count}")
print(
    f"Estimated duration: "
    f"{original_script.estimated_total_seconds} seconds"
)

## Load the editor prompt

In [ ]:
editor_system_prompt = load_prompt("script_editor")

print("Editor prompt loaded.")

## Edit the script

In [ ]:
edited_script = edit_script(
    script=original_script,
    system_prompt=editor_system_prompt,
    target_wpm=TARGET_WORDS_PER_MINUTE,
    minimum_seconds=MINIMUM_SECONDS,
    maximum_seconds=MAXIMUM_SECONDS,
    temperature=TEMPERATURE,
    seed=GENERATION_SEED,
)

print(f"Edited words: {edited_script.word_count}")
print(
    f"Edited duration: "
    f"{edited_script.estimated_total_seconds} seconds"
)

## Save the edited script

In [ ]:
output_path = (
    EDITED_SCRIPTS_DIRECTORY
    / build_edited_script_filename(edited_script)
)

save_script(
    script=edited_script,
    output_path=output_path,
)

print(f"Saved edited script to: {output_path}")

## Compare before and after

In [ ]:
changes = summarize_changes(
    original=original_script,
    edited=edited_script,
)

for name, value in changes.items():
    print(f"{name}: {value}")

## Preview the edited structure

In [ ]:
print(f"TITLE: {edited_script.topic.title}")
print()

print(f"HOOK ({edited_script.hook.estimated_seconds}s)")
print(edited_script.hook.narration)
print()

for index, section in enumerate(edited_script.sections, start=1):
    print(
        f"{index}. {section.segment_type.upper()} "
        f"({section.estimated_seconds}s)"
    )
    print(section.narration)
    print()

print(f"CLOSING ({edited_script.closing.estimated_seconds}s)")
print(edited_script.closing.narration)
print()

print(f"WORD COUNT: {edited_script.word_count}")
print(
    f"ESTIMATED TOTAL: "
    f"{edited_script.estimated_total_seconds} seconds"
)

## Preview complete narration

In [ ]:
print(edited_script.full_narration)